In [ ]:
!pip install -q -U google-genai sentence-transformers chromadb langchain-text-splitters pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 834.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 481.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import os
from google.colab import userdata, files
from google import genai
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

In [ ]:
import os
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY secret not found. Check the secret name in Colab.")

os.environ["GOOGLE_API_KEY"] = api_key

client = genai.Client(api_key=api_key)

print("Gemini API configured successfully!")

Gemini API configured successfully!


In [ ]:
from google.colab import files
from pypdf import PdfReader

print("Please upload one or more PDF files:")

uploaded = files.upload()

pdf_texts = []

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        reader = PdfReader(filename)
        text = ""

        for page_num, page in enumerate(reader.pages):

            page_text = page.extract_text()

            if page_text:
                text += f"\n--- Page {page_num + 1} ---\n"
                text += page_text

        pdf_texts.append(text)

        print(f"Loaded '{filename}' ({len(reader.pages)} pages).")

if not pdf_texts:
    raise ValueError(
        "No valid PDF files uploaded. Please re-run and upload a .pdf file."
    )

full_pdf_content = "\n\n".join(pdf_texts)

print(f"\nTotal PDFs loaded: {len(pdf_texts)}")
print(f"Total extracted characters: {len(full_pdf_content)}")

Please upload one or more PDF files:


Saving 99_BDAV P5.pdf to 99_BDAV P5.pdf
Loaded '99_BDAV P5.pdf' (4 pages).

Total PDFs loaded: 1
Total extracted characters: 596


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(full_pdf_content)

print(f"Extracted and split document into {len(chunks)} text chunks.")

Extracted and split document into 2 text chunks.


In [ ]:
print("Number of chunks:", len(chunks))

if len(chunks) > 0:
    print("First chunk:")
    print(chunks[0][:500])
else:
    print("❌ chunks is empty!")

Number of chunks: 2
First chunk:
--- Page 1 ---
Name : Akanksha Sandeep Netkar  Roll no. 99 
 
 
PRACTICAL NO. 5 
AIM : Apache Pig List of Commands. 
1. Check whether Apache Pig is installed.  
                  Command : pig –version 
 
2. Create a Input File 
 
 
 
 

--- Page 2 ---
Name : Akanksha Sandeep Netkar  Roll no. 99 
 
 
3. Verify the File 
 
4. Start Apache Pig 
 
 
5. LOAD COMMAND & DUMP COMMAND 
 
 

--- Page 3 ---
Name : Akanksha Sandeep Netkar  Roll no. 99 
 
 
6. FILTER COMMAND 
 
 
7. DISTINCT COMMAND


In [ ]:
print("Loading embedding model and building vector index...")

# Check chunks first
print("Number of chunks:", len(chunks))

if not chunks:
    raise ValueError(
        "❌ No PDF chunks found. Please check the PDF loading "
        "and text chunking step before running vector indexing."
    )

embedder = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()

# Reset collection for clean execution
try:
    chroma_client.delete_collection(name="pdf_rag_collection")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="pdf_rag_collection"
)

# Embed chunks
chunk_embeddings = embedder.encode(
    chunks,
    show_progress_bar=True
).tolist()

chunk_ids = [
    f"doc_chunk_{i}"
    for i in range(len(chunks))
]

collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=chunk_ids
)

print("PDF Vector Indexing Complete!")
print("Total chunks indexed:", len(chunks))

Loading embedding model and building vector index...
Number of chunks: 2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PDF Vector Indexing Complete!
Total chunks indexed: 2


In [ ]:
def retrieve_pdf_context(query, top_k=3):
    query_embedding = embedder.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=min(top_k, collection.count())
    )

    documents = results.get("documents", [[]])[0]

    return documents

In [ ]:
def ask_pdf(query: str):

    context_passages = retrieve_pdf_context(query, top_k=3)

    context_str = "\n".join(
        f"- {p}" for p in context_passages
    )

    prompt = f"""You are an intelligent document analysis assistant.
Answer the question using ONLY the provided PDF context below.

If the information is not contained within the provided context, state clearly:
"I cannot find the answer in the provided PDF."

PDF CONTEXT:

{context_str}

Question: {query}

Answer:"""

    try:
        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )

        return response.text, context_passages

    except Exception as e:
        print("\n⚠️ Gemini API Error:")
        print(e)

        return (
            "Gemini is currently unavailable. Please check your Gemini API "
            "key and Google Cloud project access.",
            context_passages
        )

In [ ]:
print("=" * 60)
print("PDF CHATBOT READY! Type your question below (or type 'exit' to quit).")
print("=" * 60)

while True:

    user_query = input("\nAsk a question about your PDF: ")

    if user_query.lower() in ["exit", "quit", "q"]:
        print("Exiting PDF Chatbot. Goodbye!")
        break

    if not user_query.strip():
        continue

    answer, context = ask_pdf(user_query)

    print("\n--- RETRIEVED PDF SNIPPETS ---")

    for i, snippet in enumerate(context, 1):
        print(f"[{i}] {snippet[:150]}...")

    print("\n--- CHATBOT RESPONSE ---")

    print(answer)

    print("-" * 60)

PDF CHATBOT READY! Type your question below (or type 'exit' to quit).

⚠️ Gemini API Error:
403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}

--- RETRIEVED PDF SNIPPETS ---
[1] --- Page 4 ---
Name : Akanksha Sandeep Netkar  Roll no. 99 
 
 
 
 
8. FOREACH COMMAND...
[2] --- Page 1 ---
Name : Akanksha Sandeep Netkar  Roll no. 99 
 
 
PRACTICAL NO. 5 
AIM : Apache Pig List of Commands. 
1. Check whether Apache Pig is in...

--- CHATBOT RESPONSE ---
Gemini is currently unavailable. Please check your Gemini API key and Google Cloud project access.
------------------------------------------------------------
